# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mukeshburdak/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Project:** Machine Learning-Based Content Refresh Prioritization Using Search Performance Analytics

This notebook creates a transparent, metadata-only review queue. It is a conservative baseline, not a model of future decline and not evidence that refreshing a page will improve performance. It evaluates how the rule aligns with the existing retrospective label only.

**Data boundary:** Use only the approved local anonymized starter file `data/raw/content_refresh_anonymized.csv`. Do not fetch the gated warehouse, upload data to third-party tools, or print raw rows/identifiers unnecessarily. Do not commit generated CSVs or row-level exports.

## 1. My rule and its reason codes

**Rule in plain language:** Put older content that has not been updated recently earlier in a human review queue. Give the highest priority to content that is both at least 180 days old and has not been updated for at least 180 days. Next prioritize content that is stale but not yet 180 days old; then content that is old but more recently updated. Other items stay in the lower-priority queue.

This deliberately uses only `content_age_days` and `days_since_last_update`. It does **not** use traffic, impressions, clicks, `trend_direction`, `trend_pct`, derived product flags, IDs, or other overlapping performance windows as score inputs. The rule is therefore easy to inspect, but may miss content that is declining while still fresh.

Reason codes:
- `old_and_stale`: age ≥ 180 days and days since update ≥ 180.
- `stale_only`: days since update ≥ 180 but age < 180.
- `old_only`: age ≥ 180 but days since update < 180.
- `neither_threshold`: neither threshold is met.
- `missing_metadata`: either required metadata field is missing; rank conservatively at the bottom pending review.

The 180-day thresholds are a transparent starting heuristic, not an empirically optimized cutoff. The label `is_declining_label = (trend_direction == "down")` is a retrospective proxy based on an observed trend; it is not a future outcome or refresh-benefit label.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

SEED = 42  # recorded for reproducibility; this rule itself has no randomness
DATA_PATH_CANDIDATES = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"),
]

DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Approved starter CSV not found. Place content_refresh_anonymized.csv at "
        "data/raw/content_refresh_anonymized.csv in a local/approved runtime. "
        "Do not substitute the gated warehouse or upload client data to a third-party service."
    )

df = pd.read_csv(DATA_PATH)
required = {
    "content_id", "client_id", "trend_direction",
    "content_age_days", "days_since_last_update"
}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"Required columns missing from the approved starter CSV: {missing}")

# Derive the retrospective proxy exactly as documented; do not use it in the score.
df = df.copy()
df["is_declining_label"] = (df["trend_direction"].astype("string").str.lower() == "down").astype("int8")

print(f"Loaded approved local file: {DATA_PATH}")
print(f"Rows: {len(df):,} | distinct clients: {df['client_id'].nunique(dropna=True):,}")
print(f"Label coverage: {df['is_declining_label'].notna().mean():.1%}")
print("Score inputs:", ["content_age_days", "days_since_last_update"])
print("Forbidden score inputs checked separately: IDs, trend_direction, trend_pct, performance windows, product flags.")

## 2. Build the ranked queue

The score is a simple priority tier: 3 for old-and-stale, 2 for stale-only, 1 for old-only, and 0 for neither threshold. Missing metadata is explicitly unscored and placed last. Ties retain source row order; IDs are not used as features or as ranking signals.

The output is written locally to `work/outputs/baseline_action_score.csv`. That row-level file should remain uncommitted and be handled under the repository's data-use rules.

In [ ]:
age = pd.to_numeric(df["content_age_days"], errors="coerce")
stale_days = pd.to_numeric(df["days_since_last_update"], errors="coerce")
metadata_missing = age.isna() | stale_days.isna()

is_old = age >= 180
is_stale = stale_days >= 180

df["reason_code"] = np.select(
    [
        metadata_missing,
        is_old & is_stale,
        is_stale & ~is_old,
        is_old & ~is_stale,
    ],
    [
        "missing_metadata",
        "old_and_stale",
        "stale_only",
        "old_only",
    ],
    default="neither_threshold",
)
priority_map = {
    "old_and_stale": 3,
    "stale_only": 2,
    "old_only": 1,
    "neither_threshold": 0,
    "missing_metadata": -1,
}
df["baseline_score"] = df["reason_code"].map(priority_map).astype(int)

action_map = {
    "old_and_stale": "Review for refresh",
    "stale_only": "Review update history",
    "old_only": "Check whether update is needed",
    "neither_threshold": "No age-based priority",
    "missing_metadata": "Verify metadata before prioritizing",
}
df["recommended_action"] = df["reason_code"].map(action_map)

confidence_map = {
    "old_and_stale": "Medium rule confidence; confirm content quality and business context",
    "stale_only": "Low-to-medium; age is below the old-content threshold",
    "old_only": "Low-to-medium; update recency is below the stale threshold",
    "neither_threshold": "Low for this heuristic; other signals may matter",
    "missing_metadata": "Low; required metadata is incomplete",
}
df["confidence_note"] = df["reason_code"].map(confidence_map)

# Stable sort preserves original order for tied scores; no identifier enters the score.
ranked = df.sort_values("baseline_score", ascending=False, kind="mergesort").copy()
ranked.insert(0, "baseline_rank", np.arange(1, len(ranked) + 1))

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
queue_path = output_dir / "baseline_action_score.csv"

# Keep the operational queue minimal; never export the raw input row.
queue_columns = [
    "baseline_rank", "content_id", "client_id", "baseline_score",
    "reason_code", "recommended_action", "confidence_note",
    "content_age_days", "days_since_last_update", "is_declining_label"
]
ranked[queue_columns].to_csv(queue_path, index=False)

print(f"Ranked {len(ranked):,} rows. Queue saved locally to {queue_path}.")
display(ranked["reason_code"].value_counts(dropna=False).rename_axis("reason_code").to_frame("count"))
print("Do not commit this row-level CSV. Commit only aggregate metrics JSON after the notebook has actually run and been reviewed.")

## 3. Top-20 review

This table is a **structured review aid**, not a claim that the content items have been editorially reviewed. It includes the retrospective label for diagnostic comparison. In an actual business workflow, a human should check page quality, business importance, seasonality, tracking coverage, and whether the item is eligible for an update before acting. The pseudonymous IDs are shown only to identify records in this local review queue; never attempt to map them back to real clients or publish careless row-level exports.

In [ ]:
def why_this_pick_might_be_wrong(reason):
    return {
        "old_and_stale": "Age and update recency alone do not prove declining demand or refresh value.",
        "stale_only": "The item is not yet 180 days old; staleness may be expected for this content type.",
        "old_only": "It was updated within the last 180 days; an additional refresh may not be warranted.",
        "neither_threshold": "A young/recently updated item can still have a real performance or quality issue.",
        "missing_metadata": "Missing age/update metadata may reflect data quality rather than content priority.",
    }.get(reason, "Rule assumptions may not fit this item.")

top20 = ranked.head(20).copy()
top20["what_would_make_it_wrong"] = top20["reason_code"].map(why_this_pick_might_be_wrong)
review_columns = [
    "baseline_rank", "content_id", "baseline_score", "recommended_action",
    "reason_code", "confidence_note", "what_would_make_it_wrong",
    "content_age_days", "days_since_last_update", "is_declining_label"
]
display(top20[review_columns])

print("Review prompt: identify at least one weak pick and explain which missing business/context signal could change its priority.")
print("No editorial review has been completed automatically; record human judgments separately and only if actually performed.")

## 4. Weak picks, baseline metrics, and leakage check

Precision@K is calculated against the retrospective decline proxy on the **same rows** as the baseline ranking. Compare it with the overall label base rate; do not interpret the difference causally. This is an in-sample descriptive diagnostic, not out-of-sample model performance and not evidence of future decline or refresh impact.

Because the label is derived from `trend_direction`, and `trend_direction` is derived from `trend_pct`, neither is allowed into the score. The rule also excludes performance windows and derived/product flags. The age metadata may still be associated with content type or editorial processes, so treat the results as a rough baseline only. A future ML-08 model should use the same target and evaluation slice only if the team confirms that target is appropriate; a future-benefit target would require a different time-aligned label.

In [ ]:
def precision_at_k(labels, k):
    labels = np.asarray(labels, dtype=float)
    if len(labels) == 0 or k <= 0:
        return float("nan")
    k = min(int(k), len(labels))
    return float(labels[:k].mean())

valid = ranked["is_declining_label"].notna()
eval_ranked = ranked.loc[valid].copy()
labels_in_rank_order = eval_ranked["is_declining_label"].astype(int).to_numpy()
base_rate = float(labels_in_rank_order.mean()) if len(labels_in_rank_order) else float("nan")
k_values = {10: min(10, len(eval_ranked)), 20: min(20, len(eval_ranked))}
if len(eval_ranked):
    # Top 10% gives a scale-sensitive review queue; floor to at least one item.
    k_values["top_10_percent"] = max(1, int(np.ceil(0.10 * len(eval_ranked))))
else:
    k_values["top_10_percent"] = 0

metrics = {
    "project": "Machine Learning-Based Content Refresh Prioritization Using Search Performance Analytics",
    "assignment": "ML-07",
    "baseline": "metadata_age_and_staleness_rule",
    "seed": SEED,
    "n_rows": int(len(df)),
    "n_evaluable_labels": int(len(eval_ranked)),
    "label_definition": "is_declining_label = (trend_direction == 'down')",
    "label_scope": "retrospective observed-trend proxy; not future decline or refresh benefit",
    "score_inputs": ["content_age_days", "days_since_last_update"],
    "excluded_inputs": [
        "content_id", "client_id", "trend_direction", "trend_pct",
        "performance windows and totals", "derived/product decision flags"
    ],
    "threshold_days": 180,
    "base_rate": base_rate,
    "precision_at_k": {
        str(k): precision_at_k(labels_in_rank_order, k)
        for k in [10, 20, k_values["top_10_percent"]]
        if k > 0
    },
    "evaluation_note": "In-sample descriptive alignment only; no causal or future-performance claim.",
}
print(f"Retrospective label base rate: {base_rate:.3f}" if len(eval_ranked) else "No evaluable labels.")
for k, value in metrics["precision_at_k"].items():
    print(f"Precision@{k}: {value:.3f}" if np.isfinite(value) else f"Precision@{k}: not available")

# Machine-readable results are created only by executing this notebook on the approved data.
metrics_path = output_dir / "ml07_baseline_metrics.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, allow_nan=False)
print(f"Aggregate metrics saved locally to {metrics_path}. Review before committing this JSON.")

In [ ]:
# Leakage guard: verify the rule's actual score dependencies and the ranking output.
score_inputs = {"content_age_days", "days_since_last_update"}
forbidden_inputs = {
    "content_id", "client_id", "trend_direction", "trend_pct",
    "is_declining_label", "impressions_90d", "clicks_90d",
    "sessions_90d", "impressions_last_30d", "clicks_last_30d",
    "sessions_last_30d", "impressions_prev_30d", "clicks_prev_30d",
    "sessions_prev_30d", "health_score", "needs_ctr_fix", "is_quick_win"
}
assert score_inputs.isdisjoint(forbidden_inputs)
assert ranked["baseline_rank"].is_monotonic_increasing
assert ranked["baseline_score"].notna().all()
assert set(ranked["reason_code"].unique()).issubset(set(priority_map))
assert "trend_direction" not in score_inputs and "trend_pct" not in score_inputs
assert "is_declining_label" not in score_inputs
print("Leakage guard passed: the rule score uses only age/update-recency metadata.")
print("Scope warning: metadata-only scoring is intentionally limited; it does not estimate refresh benefit.")
print("Weak-pick review still requires a human. Do not mark ML-07 complete until Run All succeeds and outputs are reviewed.")

## Self-check

- [ ] Ran all cells top to bottom in an approved environment.
- [ ] Confirmed the local anonymized starter CSV and schema; no gated warehouse used.
- [ ] Reviewed the top-20 queue and documented at least one weak pick based on actual inspection.
- [ ] Compared precision@K with the label base rate and described both as retrospective/in-sample.
- [ ] Confirmed no IDs, `trend_direction`, `trend_pct`, performance windows, or product flags enter the score.
- [ ] Kept row-level CSV local/uncommitted; committed only aggregate metrics JSON generated by a real run.
- [ ] Made no causal claims and did not claim to predict future decline or refresh benefit.

**Status:** Notebook logic prepared, but results are pending until it is run against the approved local dataset and reviewed.